# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [49]:
import os
import sys

import re
from typing import List

import pandas as pd

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'whisper_transcripts')
transcripts = os.listdir(yt_data_path)

In [4]:
dfs = []

for transcript in tqdm(transcripts):
    transcript_path = os.path.join(yt_data_path, transcript)
    df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
    dfs.append(df)

raw_transcripts_df = DataProcessing.concat_dfs(dfs)
raw_transcripts_df

100%|██████████| 7/7 [00:00<00:00, 94.73it/s]


,Text,Start Time,Duration,Video ID
0,"We all know March Madness is all about filling out and following your bracket,",0.00,3.22,-rjnvL9LL3U
1,"so we offer some first-round advice. Since UMBC beat Virginia in 2018,",3.36,4.30,-rjnvL9LL3U
2,"the 16th seed has won two of 24 meetings versus the one. Occasional upset aside,",7.86,5.06,-rjnvL9LL3U
3,"though, the analytics say advance all the one, two, and three seeds in your bracket.",12.98,3.84,-rjnvL9LL3U
4,"Looking for a long shot? The 11th seed has won half its meetings with the sixth seed of late,",17.18,4.12,-rjnvL9LL3U
...,...,...,...,...
1440,We've got all kinds of great coverage still coming up.,19.72,0.32,ZZN7BAYeOtc
1441,Thanks to everybody for being here.,21.44,0.16,ZZN7BAYeOtc
1442,"Thanks to our incredible staff back in Bristol, Connecticut.",24.26,0.24,ZZN7BAYeOtc
1443,We love you guys.,25.32,0.32,ZZN7BAYeOtc


In [ ]:
def clean_data(df) -> list[str]:
    """
    Split running text into sentence-like segments.
    Rules:
    - Split only where a period (.) or question mark (?) is immediately followed by
    whitespace. This avoids splitting decimals like ``4.5`` (digit after the dot,
    not a space).
    - Leading/trailing whitespace is stripped from each segment.
    - Empty segments are dropped.
    Caveat:
    - Abbreviations such as ``Mr. Smith`` still match ". " and may split incorrectly;
    handle those with a tokenizer or an allowlist if needed.
    Parameters
    ----------
    text : dataframe
        Full text to segment (e.g. one block of joined transcript lines).
    Returns
    -------
    list of str
        Non-empty strings, each intended as one sentence or clause.
    """

    text_joined = ' '.join(df.Text.to_list())
    raw_parts = re.split(r'(?<=[.?])\s+', text_joined)
    text_split = [p.strip() for p in raw_parts if p.strip()]

    return text_split

In [55]:
cleaned_transcripts = clean_data(raw_transcripts_df)
cleaned_transcripts

['We all know March Madness is all about filling out and following your bracket, so we offer some first-round advice.',
 'Since UMBC beat Virginia in 2018, the 16th seed has won two of 24 meetings versus the one.',
 'Occasional upset aside, though, the analytics say advance all the one, two, and three seeds in your bracket.',
 'Looking for a long shot?',
 'The 11th seed has won half its meetings with the sixth seed of late, and the ninth seed has won two-thirds of the games against the eighth seed.',
 'Now, you all had UConn winning it last year.',
 'Clark, though, had both finalists.',
 'So, who makes it to the Final Four this time around?',
 'Who cuts down the nets?',
 "You're our clubhouse leader.",
 'Oh, so I get to choose first?',
 'You get to go first.',
 "Well, I tell you, I'm going to take Michigan State, the regular season champs in the Big Ten.",
 'I take them out of the South.',
 "Florida's the most complete team in the field, in my opinion.",
 'I like them coming out of the

In [83]:
cleaned_transcripts[780:810]

["So I'm going the under, and I'm going to go 23-20 Seattle.",
 'Oh, cover by the Patriots.',
 'Good cover.',
 'Patriots plus four and a half.',
 "That's the Madden prediction, by the way.",
 'I did not know that.',
 "Yeah, I would not have known that either, but my table's right next to them, and they had it up on the screen.",
 "We're doing a lot of talking about this.",
 "So here's favored by three and a half to six and a half point in the Super Bowl.",
 'Last five instances, says Hembo.',
 'Obviously, Rams, Patriots, Panthers, and Niners.',
 'In those five games, favorites went 1-4 outright and 0-5 against the spread.',
 'Seahawks sit there at favored by 4.5.',
 'Feels like we have three Patriot covers here by the toxic table into tone.',
 'Dogs have covered five straight and 14 of the last 18, I believe.',
 'Two weeks to prepare for a team.',
 'D-Butt, you were drafted to the New England Patriots.',
 'You thought about putting your bandwagon behind the New England Patriots.',
 "Ha

## Prompt LLM

### Create Prompt

In [ ]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.
    - If the sentence only asks who/what/whether and does not state an outcome, it is not a prediction.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [85]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'granite-3.3-8b-instruct'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x17ab9ae2c50>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x17aba20a550>}

### Pass data + prompt into each LLM

In [98]:
cleaned_transcripts = cleaned_transcripts[329:]

In [ ]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: We all know March Madness is all about filling out and following your bracket, so we offer some first-round advice.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
1 --- Sentence: Since UMBC beat Virginia in 2018, the 16th seed has won two of 24 meetings versus the one.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
2 --- Sentence: Occasional upset aside, though, the analytics say advance all the one, two, and three seeds in your bracket.
 Model: llama-3.1-8b-instant | Label: 1
 Model: llama-3.3-70b-versatile | Label: 1
3 --- Sentence: Looking for a long shot?
4 --- Sentence: The 11th seed has won half its meetings with the sixth seed of late, and the ninth seed has won two-thirds of the games against the eighth seed.
5 --- Sentence: Now, you all had UConn winning it last year.
6 --- Sentence: Clark, though, had both finalists.
7 --- Sentence: So, who makes it to the Final Four this time around?
8 --

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn2rg8gf1jax2p86k7vdraq` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99969, Requested 222. Please try again in 2m45.024s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [87]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
651,Is that your prop?,"{""label"": 0}",0,None,llama-3.3-70b-versatile
652,"That's not my best bet, though.","{""label"": 0}",0,None,llama-3.1-8b-instant
653,"That's not my best bet, though.","{""label"": 0}",0,None,llama-3.3-70b-versatile
654,But you can give a prop.,"{""label"": 0}",0,None,llama-3.1-8b-instant
655,But you can give a prop.,"{""label"": 0}",0,None,llama-3.3-70b-versatile
656,"Well, there's a prop.","{""label"": 0}",0,None,llama-3.1-8b-instant


In [88]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile
0,"21 years, he's getting very bold.",0.0,0.0
1,"29 carries in the second half against the Denver Broncos, that's awesome.",0.0,0.0
2,A little less bold.,0.0,0.0
3,A week from this Sunday.,0.0,0.0
4,Absolutely nothing.,0.0,0.0
...,...,...,...
312,came down to a field goal by Adam Vinatieri to win that Super Bowl.,0.0,0.0
313,like eight of the top ten are Seahawks?,0.0,0.0
314,the Pats go up 23 to 20.,0.0,0.0
315,this way.,0.0,0.0


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [89]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
0,"21 years, he's getting very bold.",0.0,0.0,0
1,"29 carries in the second half against the Denver Broncos, that's awesome.",0.0,0.0,0
2,A little less bold.,0.0,0.0,0
3,A week from this Sunday.,0.0,0.0,0
4,Absolutely nothing.,0.0,0.0,0
...,...,...,...,...
312,came down to a field goal by Adam Vinatieri to win that Super Bowl.,0.0,0.0,0
313,like eight of the top ten are Seahawks?,0.0,0.0,0
314,the Pats go up 23 to 20.,0.0,0.0,0
315,this way.,0.0,0.0,0


In [90]:
results_df[results_df['Majority Vote Label']==1]

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
22,"And at the end of the day, Super Bowl 60 is going to be won by the Seattle Seahawks.",1.0,1.0,1
25,"And he has been vocal quite a bit about Seattle being able to lay in wait and destroy teams, and that's where I'm going.",1.0,1.0,1
29,And it's going to be a 24-10 victory for the Seattle Seahawks.,1.0,1.0,1
32,And that's my prediction for Super Bowl 60 in advance.,1.0,1.0,1
35,"And the Blue Devils, assuming the Cooper flag is healthy, I think, are cutting down the in San Antonio.",1.0,1.0,1
36,And the MVP is going to be?,1.0,1.0,1
39,And their defense is going to do it.,1.0,1.0,1
42,"And they are going to get the defensive stops, 24-10.",1.0,1.0,1
45,"And your Super Bowl MVP will be Demarcus Lawrence, baby.",1.0,1.0,1
59,But this is going to come down to an Andy Borogalis game-winning field goal.,1.0,1.0,1


In [99]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "majority_vote", "sports")
DataProcessing.save_to_file(results_df, path=save_data_path, prefix='batch_1', save_file_type='csv', include_version=False)

Saving CSV file to: c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\prediction_acquition-youtube\../data\yt\majority_vote\sports\batch_1.csv
